In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt  
import seaborn as sns  # ← И ЭТУ (для красивой визуализации)

In [2]:
class PipeDataset(Dataset):
    def __init__(self, data_list, labels, scaler=None, fit_scaler=False):
        """
        data_list: список numpy массивов формы (length, 8)
        labels: список меток классов (int)
        """
        self.data_list = data_list
        self.labels = labels
        
        # Нормализация по всему датасету (важно!)
        if fit_scaler:
            # Объединяем все данные для расчета статистик
            all_data = np.vstack(data_list)  # (total_points, 8)
            self.scaler = StandardScaler().fit(all_data)
        else:
            self.scaler = scaler
            
        # Применяем нормализацию к каждому образцу
        self.data_list = [
            torch.FloatTensor(self.scaler.transform(sample)) 
            for sample in self.data_list
        ]
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        return self.data_list[idx], self.labels[idx]

def collate_fn(batch):
    """
    Формирует батч из данных переменной длины.
    Дополняет (pad) короткие последовательности нулями до самой длинной в батче.
    """
    data_list, labels = zip(*batch)
    
    # Находим максимальную длину в текущем батче
    max_len = max([d.shape[0] for d in data_list])
    
    # Создаем тензоры с паддингом
    padded_data = []
    lengths = []
    
    for data in data_list:
        length = data.shape[0]
        lengths.append(length)
        
        # Дополняем нулями до max_len: (length, 8) -> (max_len, 8)
        padding = torch.zeros(max_len - length, data.shape[1])
        padded = torch.cat([data, padding], dim=0)
        padded_data.append(padded)
    
    # Стек: (Batch, Max_Len, 8)
    data_batch = torch.stack(padded_data)
    labels_batch = torch.stack(labels)
    
    # Создаем маску для CNN (чтобы игнорировать паддинг при пулинге)
    # 1 = реальные данные, 0 = паддинг
    mask = torch.zeros(data_batch.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1
    
    return data_batch, labels_batch, mask, torch.tensor(lengths)

In [3]:
class Pipe1DCNN(nn.Module):
    def __init__(self, input_channels=8, n_classes=3, dropout=0.3):
        super().__init__()
        
        self.features = nn.Sequential(
            # Блок 1
            nn.Conv1d(input_channels, 32, kernel_size=5, padding=2), # padding='same'
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),  # Уменьшаем длину в 2 раза
            
            # Блок 2
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),  # Уменьшаем длину в 2 раза
            
            # Блок 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            
            # АДАПТИВНЫЙ ПУЛИНГ - ключ к переменной длине!
            # Превращает (Batch, 128, Any_Length) -> (Batch, 128, 1)
            nn.AdaptiveAvgPool1d(1)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    
    def forward(self, x, mask=None):
        # Вход: (Batch, Length, 8) -> для Conv1d нужно (Batch, 8, Length)
        x = x.transpose(1, 2)
        x = self.features(x)  # (Batch, 128, 1)
        x = self.classifier(x)
        return x

In [4]:
class PipeCNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(8, 32, 9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.block = nn.Sequential(
            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.attention = nn.Sequential(
            nn.Conv1d(64, 32, 1),
            nn.ReLU(),
            nn.Conv1d(32, 1, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)

        x = self.stem(x)
        x = self.block(x)

        w = self.attention(x)
        w = torch.softmax(w, dim=2)

        x = (x * w).sum(dim=2)  # (B, 64)

        return x

In [5]:
class ImprovedPipeCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(8, 32, 9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.block1 = ResidualBlock(32)
        self.block2 = ResidualBlock(32)

        self.down = nn.Conv1d(32, 64, 3, stride=2, padding=1)

        self.block3 = ResidualBlock(64)
        self.block4 = ResidualBlock(64)

        self.attention = nn.Sequential(
            nn.Conv1d(64, 32, 1),
            nn.ReLU(),
            nn.Conv1d(32, 1, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = x.transpose(1, 2)

        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)

        x = self.down(x)

        x = self.block3(x)
        x = self.block4(x)

        w = self.attention(x)
        w = torch.softmax(w, dim=2)

        x = (x * w).sum(dim=2)

        return self.classifier(x)

In [6]:
import torch
import numpy as np

def train_epoch(model, classifier, loader, criterion, optimizer, device, scaler):
    model.train()
    classifier.train()
    total_loss, correct, total = 0, 0, 0

    for data, labels, lengths in loader:
        data, labels = data.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Смешанная точность (автовыбор устройства)
        with torch.autocast(device_type=device.type):
            features = model(data)       # CNN → embedding (B, 64)
            outputs = classifier(features)
            loss = criterion(outputs, labels)

        # Gradient scaling для mixed precision
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * data.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / total, 100. * correct / total


def evaluate(model, classifier, loader, criterion, device):
    model.eval()
    classifier.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for data, labels, lengths in loader:
            data, labels = data.to(device), labels.to(device)
            
            with torch.autocast(device_type=device.type):
                features = model(data)
                outputs = classifier(features)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * data.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, 100. * correct / total, all_preds, all_labels

In [7]:
import struct
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
import os

def read_raw_file(filepath):
    """Читает .raw файл и возвращает данные в формате для нейросети"""
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"❌ Файл не найден: {path}")

    with open(path, 'rb') as f:
        data = f.read()

    if len(data) < 17:
        raise ValueError("⚠️ Файл слишком короткий")

    # Парсинг заголовка (Big-Endian)
    x = struct.unpack_from('>i', data, 0)[0]  # длина (например, 202)
    y = struct.unpack_from('>i', data, 4)[0]  # ширина (8)
    thicknom = struct.unpack_from('>d', data, 8)[0]
    defective = struct.unpack_from('>B', data, 16)[0]

    # Извлечение матрицы
    num_values = x * y
    matrix = struct.unpack(f'>{num_values}d', data[17:17+num_values*8])
    
    # Преобразование в numpy массив и ресейп в (length, 8)
    matrix_np = np.array(matrix, dtype=np.float32).reshape(x, y)
    
    return {
        'matrix': matrix_np,        # Форма: (202, 8)
        'defective': defective,     # 0 или 1
        'thicknom': thicknom,
        'filename': path.name
    }

def load_dataset_from_folder(folder_path, verbose=True):
    """Загружает все .raw файлы из папки"""
    folder = Path(folder_path)
    raw_files = list(folder.glob('*.raw'))
    
    if not raw_files:
        raise FileNotFoundError(f"❌ Не найдено .raw файлов в {folder_path}")
    
    if verbose:
        print(f"📁 Найдено файлов: {len(raw_files)}")
    
    data_list = []
    labels = []
    filenames = []
    thicknom_list = []  # ← Добавляем список thicknom
    
    good_count = 0
    bad_count = 0
    
    for file_path in raw_files:
        try:
            result = read_raw_file(file_path)
            data_list.append(result['matrix'])
            labels.append(result['defective'])
            filenames.append(result['filename'])
            thicknom_list.append(result['thicknom'])  # ← Сохраняем thicknom
            
            if result['defective'] == 0:
                good_count += 1
            else:
                bad_count += 1
                
            if verbose and len(data_list) % 10 == 0:
                print(f"  Загружено: {len(data_list)}/{len(raw_files)}")
                
        except Exception as e:
            print(f"⚠️ Ошибка при чтении {file_path.name}: {e}")
            continue
    
    if verbose:
        print(f"\n✅ Загружено: {len(data_list)} файлов")
        print(f"   🟢 Годных (0): {good_count}")
        print(f"   🔴 Бракованных (1): {bad_count}")
        print(f"   📊 Размер матрицы: {data_list[0].shape}")
    
    return data_list, np.array(labels), filenames, thicknom_list  # ← Возвращаем thicknom_list

def create_train_val_split(data_list, labels, thicknom_list, test_size=0.2, random_state=42):
    """Разделяет данные на тренировочные и валидационные наборы"""
    
    X_train, X_val, y_train, y_val, thicknom_train, thicknom_val = train_test_split(
        data_list, 
        labels, 
        thicknom_list,  # ← Добавляем разделение thicknom
        test_size=test_size, 
        random_state=random_state,
        stratify=labels  # Сохраняем баланс классов
    )
    
    print(f"\n📊 РАЗМЕРЫ ДАТАСЕТА:")
    print(f"   Train: {len(X_train)} образцов")
    print(f"   Val:   {len(X_val)} образцов")
    print(f"   Train классы: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
    print(f"   Val классы:   0={sum(y_val==0)}, 1={sum(y_val==1)}")
    
    return X_train, X_val, y_train, y_val, thicknom_train, thicknom_val

# ============================================
# 🚀 ЗАПУСК ПОДГОТОВКИ ДАННЫХ
# ============================================

# Путь к папке с данными
DATASET_PATH = 'dataset'  # или полный путь 'D:/Универ/ВКР/Код/ПарсингДанных/DataTransform/scientificProject/dataset'

print("="*60)
print("🔄 ЗАГРУЗКА ДАННЫХ ИЗ .raw ФАЙЛОВ")
print("="*60)

# Загрузка всех файлов
data_list, labels, filenames, thicknom_list = load_dataset_from_folder(DATASET_PATH)

# Разделение на train/val
X_train, X_val, y_train, y_val, thicknom_train, thicknom_val = create_train_val_split(
    data_list, 
    labels, 
    thicknom_list,  # ← Добавляем разделение thicknom
    test_size=0.2,  # 20% на валидацию
    random_state=42
)

print("\n" + "="*60)
print("✅ ДАННЫЕ ГОТОВЫ ДЛЯ ОБУЧЕНИЯ!")
print("="*60)

# Теперь эти переменные можно использовать в вашем коде:
# X_train: список numpy массивов [(202, 8), (198, 8), ...]
# y_train: numpy массив меток [0, 1, 0, 1, ...]
# X_val, y_val: валидационные данные

🔄 ЗАГРУЗКА ДАННЫХ ИЗ .raw ФАЙЛОВ
📁 Найдено файлов: 3486
  Загружено: 10/3486
  Загружено: 20/3486
  Загружено: 30/3486
  Загружено: 40/3486
  Загружено: 50/3486
  Загружено: 60/3486
  Загружено: 70/3486
  Загружено: 80/3486
  Загружено: 90/3486
  Загружено: 100/3486
  Загружено: 110/3486
  Загружено: 120/3486
  Загружено: 130/3486
  Загружено: 140/3486
  Загружено: 150/3486
  Загружено: 160/3486
  Загружено: 170/3486
  Загружено: 180/3486
  Загружено: 190/3486
  Загружено: 200/3486
  Загружено: 210/3486
  Загружено: 220/3486
  Загружено: 230/3486
  Загружено: 240/3486
  Загружено: 250/3486
  Загружено: 260/3486
  Загружено: 270/3486
  Загружено: 280/3486
  Загружено: 290/3486
  Загружено: 300/3486
  Загружено: 310/3486
  Загружено: 320/3486
  Загружено: 330/3486
  Загружено: 340/3486
  Загружено: 350/3486
  Загружено: 360/3486
  Загружено: 370/3486
  Загружено: 380/3486
  Загружено: 390/3486
  Загружено: 400/3486
  Загружено: 410/3486
  Загружено: 420/3486
  Загружено: 430/3486
  Загру

In [13]:
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.ndimage import label

def extract_features(matrix, thicknom):
    m = matrix.copy()
    
    # ⚠️ убираем нулевой padding (очень важно!)
    mask = m > 0
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        m = m[rows[0]:rows[-1]+1]

    # нормализация относительно номинала
    m = m / thicknom
    
    features = []
    
    # ======================
    # 📊 1. Базовая статистика
    # ======================
    flat = m.flatten()
    
    features += [
        flat.min(),
        flat.max(),
        flat.mean(),
        flat.std(),
        skew(flat),
        kurtosis(flat)
    ]
    
    # ======================
    # 📉 2. Дефектные зоны
    # ======================
    threshold = 0.9  # 90% от номинала
    defect_mask = m < threshold
    
    defect_ratio = defect_mask.mean()
    features.append(defect_ratio)
    
    # ======================
    # 🧱 3. Кластеры дефектов
    # ======================
    labeled, num_clusters = label(defect_mask)
    features.append(num_clusters)
    
    # ======================
    # 📈 4. Градиенты
    # ======================
    gx, gy = np.gradient(m)
    grad = np.sqrt(gx**2 + gy**2)
    
    features += [
        grad.mean(),
        grad.max()
    ]
    
    # ======================
    # 📏 5. Профиль трубы
    # ======================
    profile = m.mean(axis=1)
    
    features += [
        profile.min(),
        profile.max(),
        profile.mean(),
        profile.std()
    ]
    
    return np.array(features, dtype=np.float32)


def build_feature_dataset(data_list, labels, thicknom_list):
    X = []
    
    for matrix, tnom in zip(data_list, thicknom_list):
        features = extract_features(matrix, tnom)
        X.append(features)
        
    return np.vstack(X), np.array(labels)


XGBoost

In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report



def train_xgboost(X_train, y_train, X_val, y_val):
    
    model = XGBClassifier(
        n_estimators=400,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        eval_metric='logloss',
        random_state=42
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=True
    )
    
    # ===== Оценка =====
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    
    print("\n📊 ОЦЕНКА МОДЕЛИ")
    print("="*50)
    
    print("ROC-AUC:", roc_auc_score(y_val, y_proba))
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred))
    
    return model

CNN + XGBoost

In [25]:
import os
from datetime import datetime
from pathlib import Path
import torch


# 🚀 Включаем оптимизации CUDA
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

# Если фиксированные размеры — ускорит сильно
torch.backends.cudnn.deterministic = False
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# ============================================
# 📁 ФУНКЦИЯ СОЗДАНИЯ ПАПКИ ДЛЯ РЕЗУЛЬТАТОВ
# ============================================



def create_results_folder(epochs, batch_size, learning_rate, model_type='CNN_XGBoost', base_dir='results'):
    """Создаёт папку для сохранения результатов эксперимента"""
    folder_name = f"{model_type}_{epochs}epochs_{batch_size}batch_{learning_rate}lr"
    results_dir = Path(base_dir) / folder_name
    results_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n📁 Папка результатов: {results_dir.absolute()}")
    return results_dir, folder_name

# ============================================
# 📊 PIPE DATASET (с нормализацией)
# ============================================

class PipeDataset(Dataset):
    def __init__(self, data_list, labels, scaler=None, fit_scaler=False):
        if fit_scaler:
            all_data = np.vstack(data_list)
            from sklearn.preprocessing import StandardScaler
            self.scaler = StandardScaler().fit(all_data)
        else:
            self.scaler = scaler
        
        self.data_list = [torch.FloatTensor(self.scaler.transform(sample)) 
                         for sample in data_list]
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        return self.data_list[idx], self.labels[idx]

def collate_fn(batch):
    """Формирует батч с padding для переменной длины"""
    data_list, labels = zip(*batch)
    max_len = max([d.shape[0] for d in data_list])
    
    padded_data = []
    lengths = []
    
    for data in data_list:
        length = data.shape[0]
        lengths.append(length)
        padding = torch.zeros(max_len - length, data.shape[1])
        padded = torch.cat([data, padding], dim=0)
        padded_data.append(padded)
    

    data_batch = torch.stack(padded_data)
    labels_batch = torch.stack(labels)
    
    # Создаем маску: 1 = реальные данные, 0 = паддинг
    mask = torch.zeros(data_batch.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1
    
    return data_batch, labels_batch, mask, torch.tensor(lengths)


🔥 GPU: CPU

📁 Папка результатов: /mnt/d/Универ/ВКР/Код/ПарсингДаных/DataTransform/scientificProject/results/1D_CNN_XGBoost_200epochs_32batch_0.001lr

✅ DataLoader создан!
   Train batches: 22
   Val batches:   6

🚀 Старт обучения на cpu
📊 Размеров параметров: 14,945
📁 Результаты будут сохранены в: results/1D_CNN_XGBoost_200epochs_32batch_0.001lr


In [26]:

# ============================================
# === ЦИКЛ ОБУЧЕНИЯ ===
# ============================================

best_val_acc = 0
patience_counter = 0
early_stop_patience = 20  # Early stopping
scaler = torch.cuda.amp.GradScaler()

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, classifier, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_acc, val_preds, val_labels = evaluate(model, classifier, val_loader, criterion, DEVICE)
    
    # Сохраняем метрики
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    scheduler.step(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), results_dir / 'best_pipe_cnn_lstm.pth')
        print(f"  ⭐ Новая лучшая модель! Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
    
    # Early Stopping
    if patience_counter >= early_stop_patience:
        print(f"\n🛑 Early stopping на эпохе {epoch+1} (нет улучшений {early_stop_patience} эпох)")
        break




def extract_cnn_features(model, loader, device):
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for data, y, mask, lengths in loader:
            data = data.to(device)

            emb = model(data)  # (B, 64)

            features.append(emb.cpu().numpy())
            labels.append(y.numpy())

    return np.vstack(features), np.concatenate(labels)



/tmp/ipykernel_12131/3935131073.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/home/strel/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/home/strel/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  ⭐ Новая лучшая модель! Acc: 82.23%
Epoch 01 | Train Loss: 0.6449 Acc: 67.65% | Val Loss: 0.6328 Acc: 82.23%
  ⭐ Новая лучшая модель! Acc: 85.82%
Epoch 02 | Train Loss: 0.5901 Acc: 83.68% | Val Loss: 0.5767 Acc: 85.82%
Epoch 03 | Train Loss: 0.5634 Acc: 82.64% | Val Loss: 0.5449 Acc: 85.82%
Epoch 04 | Train Loss: 0.5401 Acc: 82.57% | Val Loss: 0.5310 Acc: 84.10%
Epoch 05 | Train Loss: 0.5210 Acc: 83.11% | Val Loss: 0.5070 Acc: 85.39%
  ⭐ Новая лучшая модель! Acc: 86.10%
Epoch 06 | Train Loss: 0.5034 Acc: 83.50% | Val Loss: 0.4890 Acc: 86.10%
Epoch 07 | Train Loss: 0.4835 Acc: 84.00% | Val Loss: 0.4624 Acc: 85.96%
  ⭐ Новая лучшая модель! Acc: 87.82%
Epoch 08 | Train Loss: 0.4648 Acc: 84.68% | Val Loss: 0.4403 Acc: 87.82%
  ⭐ Новая лучшая модель! Acc: 88.11%
Epoch 09 | Train Loss: 0.4509 Acc: 85.58% | Val Loss: 0.4265 Acc: 88.11%
Epoch 10 | Train Loss: 0.4404 Acc: 85.87% | Val Loss: 0.4350 Acc: 87.39%
Epoch 11 | Train Loss: 0.4259 Acc: 86.08% | Val Loss: 0.4300 Acc: 87.54%
  ⭐ Новая лу

In [28]:
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Извлечение CNN-эмбеддингов
# ==========================================
def extract_cnn_embeddings(model, loader, device):
    model.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for data, lbl, _, _ in loader:  # ← игнорируем mask и lengths
            data = data.to(device)
            emb = model(data)  # (Batch, 64)
            embeddings.append(emb.cpu().numpy())
            labels.extend(lbl.numpy())
    return np.vstack(embeddings), np.array(labels)

# Извлекаем эмбеддинги для Train/Val
print("📦 Извлечение CNN-эмбеддингов...")
X_train_cnn, y_train = extract_cnn_embeddings(model, train_loader, DEVICE)
X_val_cnn,   y_val   = extract_cnn_embeddings(model, val_loader, DEVICE)

# ==========================================
# 2. Формирование ручных признаков
# ==========================================
print("🔍 Формирование ручных признаков...")
# Используем thicknom_train и thicknom_val из разделения данных
X_train_feat, _ = build_feature_dataset(X_train, y_train, thicknom_train)
X_val_feat,   _ = build_feature_dataset(X_val,   y_val,   thicknom_val)

# Конкатенация: [Ручные признаки (N) | CNN-эмбеддинги (64)]
X_train_final = np.hstack([X_train_feat, X_train_cnn])
X_val_final   = np.hstack([X_val_feat,   X_val_cnn])

print(f"✅ Итоговая размерность признаков: {X_train_final.shape[1]}")

# ==========================================
# 3. Обучение XGBoost
# ==========================================
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score

# Автоматический вес класса для несбалансированных данных
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

print("🚀 Обучение XGBoost...")
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    seed=42,
    early_stopping_rounds=30
)

xgb_model.fit(
    X_train_final, y_train,
    eval_set=[(X_val_final, y_val)],
    verbose=50
)

# ==========================================
# 4. Оценка и визуализация
# ==========================================
y_pred_proba = xgb_model.predict_proba(X_val_final)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print("\n📊 Classification Report (XGBoost):")
print(classification_report(y_val, y_pred, target_names=['Годный (0)', 'Брак (1)']))
print(f"ROC-AUC: {roc_auc_score(y_val, y_pred_proba):.4f}")

# Матрица ошибок
cm = np.array([[sum((y_val==0)&(y_pred==0)), sum((y_val==0)&(y_pred==1))],
               [sum((y_val==1)&(y_pred==0)), sum((y_val==1)&(y_pred==1))]])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Годный', 'Брак'], yticklabels=['Годный', 'Брак'])
plt.xlabel('Предсказано')
plt.ylabel('Фактически')
plt.title('🎯 Confusion Matrix (LightGBM)')
plt.tight_layout()
plt.show()

📦 Извлечение CNN-эмбеддингов...
🔍 Формирование ручных признаков...


NameError: name 'thicknom_list' is not defined